# Part 5 — Scoring guesses: likelihood, KL, and the ELBO

_Rigorous Courses · Diffusion Models — Part 5 of 12_

**How to grade a probability rule against data, measure the mismatch between two rules, and compute a floor under a number you cannot reach**

In this notebook you will plot a log-likelihood curve and watch its peak land exactly on the sample mean, verify the Gaussian KL formula against brute-force numerical integration, check Jensen's inequality by simulation, and build the smallest possible latent-variable model — small enough that you can compute the ELBO for every possible guess and watch its gap to the true log-likelihood trace out a KL divergence, point for point.

---

This notebook accompanies the lesson. Run cells top to bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## Likelihood and the maximum-likelihood estimate

The lesson scored knob settings of the menu $\mathcal{N}(\mu, 1)$ against the dataset $\{1.0,\ 2.0,\ 6.0\}$ by hand. Here we let the computer do the same scoring for every $\mu$ at once and check the hand results.

### Step 1 — Score two knob settings on the tiny dataset

The dataset has three points. The model menu is every bell curve $\mathcal{N}(\mu, 1)$ — one knob, $\mu$. Because the points are independent, densities multiply, so log-densities **add**: the log-likelihood is `gaussian_logpdf(data, mu, 1.0).sum()`.

The lesson computed by hand that $\mu = 3$ beats $\mu = 0$ by a factor of about 730,000. Watch the same numbers appear.

In [ ]:
data = np.array([1.0, 2.0, 6.0])

def gaussian_logpdf(x, mu, sigma):
    z = (x - mu) / sigma
    return -0.5 * np.log(2 * np.pi * sigma ** 2) - 0.5 * z ** 2

loglik_mu0 = gaussian_logpdf(data, 0.0, 1.0).sum()
loglik_mu3 = gaussian_logpdf(data, 3.0, 1.0).sum()
likelihood_ratio = np.exp(loglik_mu3 - loglik_mu0)

print(f"log-likelihood at mu = 0: {loglik_mu0:.4f}")
print(f"log-likelihood at mu = 3: {loglik_mu3:.4f}")
print(f"mu = 3 explains the data {likelihood_ratio:,.0f} times better")

assert loglik_mu3 > loglik_mu0

### Step 2 — Sweep the knob: the peak sits on the sample mean

The lesson's slope-zero derivation proved the top of the log-likelihood hill is at $\hat\mu = \frac{1}{n}\sum_i x^{(i)}$ — the plain average, which is $3.0$ here. Now sweep $\mu$ over a fine grid, plot the whole hill, and check that its peak is exactly where the algebra said.

Look for: one clean hill (a downward parabola plus a constant), with the red dashed line — the sample mean — passing through its highest point.

In [ ]:
mu_grid = np.linspace(-1.0, 7.0, 8001)
loglik_curve = np.array([gaussian_logpdf(data, m, 1.0).sum() for m in mu_grid])
best_mu = mu_grid[np.argmax(loglik_curve)]
sample_mean = data.mean()

plt.figure(figsize=(7, 4))
plt.plot(mu_grid, loglik_curve, color="#4ea1ff")
plt.axvline(sample_mean, color="#ff7b72", linestyle="--", label=f"sample mean = {sample_mean:.1f}")
plt.xlabel("mu (the knob)")
plt.ylabel("log-likelihood (nats)")
plt.title("log-likelihood of mu for the dataset {1, 2, 6}")
plt.legend()
plt.show()

print(f"argmax of the curve: {best_mu:.3f}")
print(f"sample mean        : {sample_mean:.3f}")

assert abs(best_mu - sample_mean) < 0.005

### Step 3 — Why logs: watch the raw product underflow

The lesson claimed that a product of thousands of small density values is unusable on a computer. Prove it: draw 5,000 points from $\mathcal{N}(0, 1)$ and evaluate the likelihood both ways. Every density value here is below $0.4$, so the raw product is below $0.4^{5000} \approx 10^{-1990}$ — far past where floating point gives out (around $10^{-308}$). The product collapses to exactly `0.0`; the sum of logs stays a perfectly ordinary number.

In [ ]:
many = rng.normal(0.0, 1.0, size=5000)
log_densities = gaussian_logpdf(many, 0.0, 1.0)
raw_product = np.prod(np.exp(log_densities))
log_sum = log_densities.sum()

print(f"raw product of 5,000 densities: {raw_product}")
print(f"sum of 5,000 log-densities    : {log_sum:.1f}")

assert raw_product == 0.0
assert np.isfinite(log_sum)

The two facts this section verified, in math:

$$ \ell(\theta) = \sum_{i=1}^{n} \log p_\theta\big(x^{(i)}\big) \qquad\text{and}\qquad \hat\mu = \frac{1}{n}\sum_{i=1}^{n} x^{(i)} $$

The log turns an underflowing product into a safe sum, and for a fixed-spread Gaussian the best center is the plain average of the data.

## Jensen's inequality

Everything about KL and the ELBO rests on one inequality: because the log curve bends downward (it is **concave**), the average of the logs never exceeds the log of the average,

$$ \mathbb{E}[\log X] \;\le\; \log \mathbb{E}[X]. $$

### Step 4 — Average of logs vs log of the average

Use the lesson's two-point example: $X$ is $1$ or $9$ with equal chances. The exact values are $\mathbb{E}[\log X] = \tfrac{1}{2}(\ln 1 + \ln 9) \approx 1.0986$ and $\log \mathbb{E}[X] = \ln 5 \approx 1.6094$. Simulate 200,000 draws and confirm both numbers — and the direction of the inequality.

In [ ]:
x_two_point = rng.choice([1.0, 9.0], size=200000)
avg_of_log = np.log(x_two_point).mean()
log_of_avg = np.log(x_two_point.mean())
exact_avg_of_log = 0.5 * np.log(1.0) + 0.5 * np.log(9.0)
exact_log_of_avg = np.log(5.0)

print(f"E[log X]  simulated: {avg_of_log:.4f}   exact: {exact_avg_of_log:.4f}")
print(f"log E[X]  simulated: {log_of_avg:.4f}   exact: {exact_log_of_avg:.4f}")

assert avg_of_log < log_of_avg
assert abs(avg_of_log - exact_avg_of_log) < 0.01
assert abs(log_of_avg - exact_log_of_avg) < 0.01

## KL divergence between Gaussians

The lesson derived, line by line,

$$ D_{\mathrm{KL}}\big(\mathcal{N}(\mu_1, \sigma_1^2) \,\big\|\, \mathcal{N}(\mu_2, \sigma_2^2)\big) = \log\frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1-\mu_2)^2}{2\sigma_2^2} - \frac{1}{2}. $$

A derivation can hide an algebra slip. The KL is also, by definition, an integral we can brute-force on a grid: $\int q(x)\,[\log q(x) - \log p(x)]\,dx$, computed as a Riemann sum (part 2's "area under the curve"). If the formula and the integral agree to many decimals across several different Gaussian pairs, the derivation survives contact with arithmetic.

### Step 5 — Closed formula vs numerical integration

Four test pairs, including the lesson's worked example $D_{\mathrm{KL}}(\mathcal{N}(1,1)\,\|\,\mathcal{N}(3,1)) = 2$ and a narrow-curves case where the answer is a large 18 nats. The grid runs from $-40$ to $40$ in 400,001 steps.

In [ ]:
def kl_gauss(mu1, s1, mu2, s2):
    spread_term = np.log(s2 / s1)
    center_term = (s1 ** 2 + (mu1 - mu2) ** 2) / (2 * s2 ** 2)
    return spread_term + center_term - 0.5

def kl_numeric(mu1, s1, mu2, s2):
    x = np.linspace(-40.0, 40.0, 400001)
    dx = x[1] - x[0]
    log_q = gaussian_logpdf(x, mu1, s1)
    log_p = gaussian_logpdf(x, mu2, s2)
    integrand = np.exp(log_q) * (log_q - log_p)
    return integrand.sum() * dx

cases = [(0.0, 1.0, 1.0, 2.0), (1.0, 1.0, 3.0, 1.0), (2.0, 0.5, 5.0, 0.5), (0.0, 2.0, 0.0, 1.0)]

for mu1, s1, mu2, s2 in cases:
    closed = kl_gauss(mu1, s1, mu2, s2)
    numeric = kl_numeric(mu1, s1, mu2, s2)
    print(f"KL( N({mu1}, {s1**2}) || N({mu2}, {s2**2}) )   closed = {closed:.6f}   numeric = {numeric:.6f}")
    assert abs(closed - numeric) < 1e-4

### Step 6 — Asymmetry: order matters

Same two curves, both directions. The forward direction averages the mismatch under the narrow $\mathcal{N}(0,1)$; the reverse averages under the wide $\mathcal{N}(1,4)$, which puts real weight far out where the narrow curve has almost nothing — huge log-ratios there inflate the cost. The lesson's hand values: $0.4431$ and $1.3069$ nats.

In [ ]:
kl_qp = kl_gauss(0.0, 1.0, 1.0, 2.0)
kl_pq = kl_gauss(1.0, 2.0, 0.0, 1.0)

print(f"KL( N(0,1) || N(1,4) ) = {kl_qp:.4f} nats")
print(f"KL( N(1,4) || N(0,1) ) = {kl_pq:.4f} nats")

assert abs(kl_qp - 0.4431) < 1e-3
assert abs(kl_pq - 1.3069) < 1e-3
assert abs(kl_qp - kl_pq) > 0.5

### Step 7 — Equal spreads: KL is squared distance between centers

The special case the whole course is heading toward: when $\sigma_1 = \sigma_2 = \sigma$,

$$ D_{\mathrm{KL}} = \frac{(\mu_1 - \mu_2)^2}{2\sigma^2}. $$

In part 8, every term of the diffusion training objective is a KL between two Gaussian denoising steps with matched spreads — so every term collapses, by this line, to a squared distance between two means. That is why the loss that trains Stable Diffusion (part 9) is a mean-squared error. Verify the collapse on the worked example.

In [ ]:
kl_full = kl_gauss(1.0, 1.0, 3.0, 1.0)
kl_shortcut = (1.0 - 3.0) ** 2 / (2 * 1.0 ** 2)

print(f"full formula gives : {kl_full:.6f} nats")
print(f"shortcut gives     : {kl_shortcut:.6f} nats")

assert abs(kl_full - 2.0) < 1e-12
assert abs(kl_full - kl_shortcut) < 1e-12

## The tiniest latent model and its ELBO

A hidden fair coin $z \in \{0, 1\}$ picks a bell curve: $z = 0$ gives $\mathcal{N}(-2, 1)$, $z = 1$ gives $\mathcal{N}(+2, 1)$. You observe only $x$. Because there are exactly two hidden cases, **everything** is computable here: the marginal $p(x)$, the true posterior, and the ELBO for every possible guess. That makes this toy the one place you can watch the ELBO's gap identity

$$ \log p_\theta(x) = \mathrm{ELBO}(q) + D_{\mathrm{KL}}\big(q(z \mid x) \,\big\|\, p_\theta(z \mid x)\big) $$

hold exactly, for every $q$ at once.

### Step 8 — Build the mixture and compute log p(x) exactly

Observe $x = 0.5$. Total probability sums over the two hidden cases: $p(x) = 0.5\,\mathcal{N}(x; -2, 1) + 0.5\,\mathcal{N}(x; 2, 1)$. The lesson's hand values: component densities $\approx 0.0175$ and $\approx 0.1295$, marginal $\approx 0.0735$, log marginal $\approx -2.6102$.

In [ ]:
x_obs = 0.5
prior_z0 = 0.5
prior_z1 = 0.5
log_like_z0 = gaussian_logpdf(x_obs, -2.0, 1.0)
log_like_z1 = gaussian_logpdf(x_obs, 2.0, 1.0)
px = prior_z0 * np.exp(log_like_z0) + prior_z1 * np.exp(log_like_z1)
log_px = np.log(px)

print(f"density of x = 0.5 under component z = 0 (center -2): {np.exp(log_like_z0):.6f}")
print(f"density of x = 0.5 under component z = 1 (center +2): {np.exp(log_like_z1):.6f}")
print(f"marginal p(x = 0.5)     = {px:.6f}")
print(f"log p(x = 0.5)          = {log_px:.6f}")

assert abs(px - 0.073523) < 1e-5
assert abs(log_px - (-2.610158)) < 1e-4

### Step 9 — The true posterior by Bayes

After seeing $x = 0.5$, which hump did the coin pick? Bayes' rule (part 2): posterior weight of $z = 1$ is its share of the marginal. The observation sits closer to the $+2$ hump, so the coin is about $88\%$ likely to have landed on $1$.

In [ ]:
r_star = prior_z1 * np.exp(log_like_z1) / px

print(f"true posterior  p(z=1 | x=0.5) = {r_star:.6f}")

assert abs(r_star - 0.880797) < 1e-4

### Step 10 — Compute the ELBO for every possible guess

A guess distribution over a two-valued coin is one number: $r = q(z{=}1 \mid x)$. The ELBO from the lesson,

$$ \mathrm{ELBO}(q) = \sum_z q(z)\,\big[\log p(z) + \log p(x \mid z) - \log q(z)\big], $$

is a two-term sum we can evaluate for every $r$ on a grid. Look for: the ELBO curve stays **below** the dashed ceiling $\log p(x)$ everywhere, and **touches** it at exactly one point — the true posterior $r^* \approx 0.881$.

In [ ]:
def elbo_of(r):
    term_z0 = (1 - r) * (np.log(prior_z0) + log_like_z0 - np.log(1 - r))
    term_z1 = r * (np.log(prior_z1) + log_like_z1 - np.log(r))
    return term_z0 + term_z1

r_grid = np.linspace(0.001, 0.999, 999)
elbo_grid = np.array([elbo_of(r) for r in r_grid])
elbo_at_posterior = elbo_of(r_star)

plt.figure(figsize=(7, 4))
plt.plot(r_grid, elbo_grid, color="#4ea1ff", label="ELBO(q)")
plt.axhline(log_px, color="#ff7b72", linestyle="--", label="log p(x)  (the ceiling)")
plt.axvline(r_star, color="#3fb950", linestyle=":", label=f"true posterior r* = {r_star:.3f}")
plt.xlabel("r = q(z=1)   (the guess)")
plt.ylabel("nats")
plt.title("the ELBO never exceeds log p(x) and touches it at the true posterior")
plt.legend()
plt.show()

print(f"highest ELBO on the grid : {elbo_grid.max():.6f}")
print(f"log p(x)                 : {log_px:.6f}")
print(f"ELBO at r = r*           : {elbo_at_posterior:.6f}")

assert np.all(elbo_grid <= log_px + 1e-9)
assert abs(elbo_at_posterior - log_px) < 1e-9

### Step 11 — The gap IS the KL to the true posterior

The gap identity says $\log p(x) - \mathrm{ELBO}(q)$ should equal $D_{\mathrm{KL}}(q \,\|\, q^*)$ where $q^*$ is the true posterior $(r^*, 1-r^*)$ — for **every** guess $r$, not only the best one. Compute both curves independently and overlay them. Look for: the dashed KL curve lies exactly on top of the gap curve; the printed disagreement is at floating-point level.

In [ ]:
gap_grid = log_px - elbo_grid
kl_grid = r_grid * np.log(r_grid / r_star) + (1 - r_grid) * np.log((1 - r_grid) / (1 - r_star))
max_disagreement = np.max(np.abs(gap_grid - kl_grid))
uniform_gap = log_px - elbo_of(0.5)

plt.figure(figsize=(7, 4))
plt.plot(r_grid, gap_grid, color="#4ea1ff", linewidth=3, label="gap = log p(x) - ELBO(q)")
plt.plot(r_grid, kl_grid, color="#ff7b72", linestyle="--", label="KL(q || true posterior)")
plt.xlabel("r = q(z=1)   (the guess)")
plt.ylabel("nats")
plt.title("the ELBO gap and the KL to the posterior are the same curve")
plt.legend()
plt.show()

print(f"largest |gap - KL| across all guesses: {max_disagreement:.2e}")
print(f"gap for the uniform guess r = 0.5    : {uniform_gap:.4f} nats")

assert max_disagreement < 1e-9
assert abs(uniform_gap - 0.4338) < 1e-3

What you have watched, in one line of math: for every guess $q$,

$$ \log p_\theta(x) - \mathrm{ELBO}(q, \theta; x) = D_{\mathrm{KL}}\big(q(z \mid x) \,\big\|\, p_\theta(z \mid x)\big) \ge 0, $$

with equality exactly at $q = p_\theta(z \mid x)$. In part 8 the guess $q$ will be the fixed forward noising chain of a diffusion model, and this identity becomes the training objective.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A coin lands heads 7 times and tails 3 times in 10 independent flips. The model says each flip lands heads with probability $\theta$. Find the maximum-likelihood estimate $\hat\theta$ — by hand with the slope-zero recipe, then confirm with a grid search.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Likelihood of the sequence: $L(\theta) = \theta^7 (1-\theta)^3$ (independent flips multiply).
- Log: $\ell(\theta) = 7\ln\theta + 3\ln(1-\theta)$ (log of a product is the sum of logs).
- Slope facts: slope of $\ln\theta$ is $1/\theta$; slope of $\ln(1-\theta)$ is $-1/(1-\theta)$ (minus because raising $\theta$ lowers $1-\theta$).
- Slope of $\ell$: $\frac{7}{\theta} - \frac{3}{1-\theta}$. Set to zero: $\frac{7}{\theta} = \frac{3}{1-\theta}$.
- Cross-multiply: $7(1-\theta) = 3\theta$, so $7 = 10\theta$ and $\hat\theta = 0.7$.

```python
theta_grid = np.linspace(0.01, 0.99, 981)
loglik = 7 * np.log(theta_grid) + 3 * np.log(1 - theta_grid)
best_theta = theta_grid[np.argmax(loglik)]
print(best_theta)   # 0.7
```

**Answer:** $\hat\theta = 7/10 = 0.7$ — the observed fraction of heads.

</details>

**Problem 2.** A model assigns density values $0.02$, $0.001$, and $0.05$ to three independent data points. Compute the log-likelihood two ways — as the log of the product and as the sum of the logs — and say why software always uses the second way.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Product: $0.02 \times 0.001 \times 0.05 = 10^{-6}$. Log of the product: $\ln 10^{-6} = -13.816$.
- Sum of logs: $\ln 0.02 + \ln 0.001 + \ln 0.05 = -3.912 - 6.908 - 2.996 = -13.816$. Same number.
- With 50,000 points of typical density $0.02$ the product is about $10^{-85000}$ — it underflows to exactly `0.0` in floating point (the format gives out near $10^{-308}$), while the sum of logs is a tame number near $-196{,}000$.

```python
values = np.array([0.02, 0.001, 0.05])
print(np.log(np.prod(values)))   # -13.8155...
print(np.log(values).sum())      # -13.8155...
```

**Answer:** $-13.816$ nats both ways; logs turn an underflowing product into a safe sum.

</details>

**Problem 3.** Let $q = \mathcal{N}(0, 1)$ and $p = \mathcal{N}(1, 4)$ (so $\sigma_2 = 2$). Compute $D_{\mathrm{KL}}(q \,\|\, p)$ and $D_{\mathrm{KL}}(p \,\|\, q)$ with the closed formula. Are they equal?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Forward: $\mu_1{=}0, \sigma_1{=}1, \mu_2{=}1, \sigma_2{=}2$: $\ln 2 + \frac{1+1}{8} - \frac{1}{2} = 0.6931 + 0.25 - 0.5 = 0.4431$ nats.
- Reverse (swap roles): $\ln\frac{1}{2} + \frac{4+1}{2} - \frac{1}{2} = -0.6931 + 2.5 - 0.5 = 1.3069$ nats.
- Not equal. The reverse direction averages under the wide curve, which puts real weight where the narrow curve has almost none — those regions carry huge log-ratios.

```python
print(kl_gauss(0.0, 1.0, 1.0, 2.0))   # 0.4431...
print(kl_gauss(1.0, 2.0, 0.0, 1.0))   # 1.3069...
```

**Answer:** $0.443$ and $1.307$ nats — KL is a mismatch cost, not a distance.

</details>

**Problem 4.** Both spreads are $\sigma^2 = 0.25$ (so $\sigma = 0.5$). Compute $D_{\mathrm{KL}}(\mathcal{N}(2, 0.25) \,\|\, \mathcal{N}(5, 0.25))$ with the equal-variance shortcut, then confirm with the full formula.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Shortcut: $\frac{(2-5)^2}{2 \times 0.25} = \frac{9}{0.5} = 18$ nats.
- Full formula: $\ln\frac{0.5}{0.5} + \frac{0.25 + 9}{0.5} - \frac{1}{2} = 0 + 18.5 - 0.5 = 18$ nats. Agreement.
- Read: the centers are $3/0.5 = 6$ standard deviations apart, and $6^2/2 = 18$ — KL grows with the square of the separation measured in spread units.

```python
print((2.0 - 5.0) ** 2 / (2 * 0.25))    # 18.0
print(kl_gauss(2.0, 0.5, 5.0, 0.5))     # 18.0
```

**Answer:** $18$ nats by both routes.

</details>

**Problem 5.** For a latent model and one observation, $\log p_\theta(x) = -1.90$ nats. Guess A has KL to the true posterior of $0.65$; guess B has $0.10$. Find both ELBOs. Which guess is better — and how could you tell without knowing $\log p_\theta(x)$ or the posterior?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Gap identity rearranged: $\mathrm{ELBO} = \log p_\theta(x) - D_{\mathrm{KL}}$.
- Guess A: $-1.90 - 0.65 = -2.55$ nats. Guess B: $-1.90 - 0.10 = -2.00$ nats.
- B is better: both floors sit under the same ceiling, so the higher floor has the smaller gap.
- Without the ceiling or the posterior: compute both ELBOs directly from their definition (needs only the joint $p_\theta(x,z)$ and each $q$ — both available) and keep the larger. The identity guarantees larger ELBO $\Leftrightarrow$ smaller KL to the posterior.

```python
elbo_a = -1.90 - 0.65
elbo_b = -1.90 - 0.10
print(elbo_a, elbo_b)   # -2.55  -2.0
```

**Answer:** $\mathrm{ELBO}_A = -2.55$, $\mathrm{ELBO}_B = -2.00$; B — compare the computable floors and pick the higher one.

</details>

**Problem 6.** Using the gap identity, explain why the ELBO equals $\log p_\theta(x)$ exactly when $q(z \mid x)$ is the true posterior $p_\theta(z \mid x)$ — and why it can never exceed $\log p_\theta(x)$ for any $q$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- The identity: $\log p_\theta(x) = \mathrm{ELBO} + D_{\mathrm{KL}}(q \,\|\, p_\theta(z \mid x))$, exact for every $q$.
- KL is never negative (Jensen), so $\mathrm{ELBO} = \log p_\theta(x) - \mathrm{KL} \le \log p_\theta(x)$: subtracting a nonnegative number cannot increase a value.
- KL is zero exactly when its two arguments are equal — so the gap closes precisely when $q = p_\theta(z \mid x)$, and stays open for every other guess.
- Step 10's plot is this argument as a picture: the curve touches the ceiling at $r^*$ and only there.

**Answer:** the gap IS the KL from the guess to the true posterior — zero iff the guess is the posterior, positive otherwise.

</details>

**Problem 7.** In diffusion models (and VAEs) we never maximize $\log p_\theta(x)$ directly — we maximize the ELBO instead. Give the two main reasons this is a sensible strategy rather than an act of desperation.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- **Computability.** $\log p_\theta(x)$ hides a sum (or integral) over every possible hidden value inside a log — for diffusion, over every possible noising path $x_1, \dots, x_T$ of a thousand continuous steps. Not evaluable. The ELBO is an average of individually computable log terms, and Monte-Carlo (part 2) estimates it by sampling a few $z$ from $q$.
- **Safety.** Because $\mathrm{ELBO} \le \log p_\theta(x)$ always, certifying the floor at some value certifies the truth is at least that value — progress on the floor is never an illusion.
- Bonus: the gap equals $D_{\mathrm{KL}}(q \,\|\, \text{true posterior})$, so raising the ELBO also means the guess is tracking the model's true belief about the hidden causes.
- Part 8 fixes $q$ to be the forward noising chain; the diffusion ELBO then splits into per-step Gaussian KLs with matched spreads, which Step 7's shortcut turns into squared errors — the trainable loss of parts 9 and 10.

**Answer:** the floor is computable where the ceiling is not, and it is safe — with a gap that is itself a meaningful KL.

</details>

## Wrap-up

Verified in this notebook: the log-likelihood hill for $\mathcal{N}(\mu, 1)$ peaks exactly at the sample mean ($3.000$ for the dataset $\{1, 2, 6\}$), and raw likelihood products underflow while log sums stay finite; Jensen's inequality $\mathbb{E}[\log X] \le \log\mathbb{E}[X]$ holds in simulation with the exact two-point values; the closed-form Gaussian KL matches brute-force integration to better than $10^{-4}$ across four cases, is visibly asymmetric, and collapses to $\frac{(\mu_1-\mu_2)^2}{2\sigma^2}$ when spreads match; and on a fully enumerable latent toy, the ELBO stayed below $\log p(x)$ for every guess, touched it exactly at the true posterior, and its gap coincided with $D_{\mathrm{KL}}(q \,\|\, \text{posterior})$ to floating-point precision.

Next, Part 6 — the forward process: the fixed noising recipe $q(x_t \mid x_{t-1})$, the survival products $\alpha_t$ and $\bar\alpha_t$, and the closed form that jumps from a clean image to any noise level in one line.